# 04a — EXP_01: No-RAG Baseline

**Architecture:** direct LLM, no retrieval. **Generator:** `llama-3.3-70b-versatile` via Groq (T=0, max_tokens=700 per Excel `Experiments Guide`). **Disk-cached** — restarts are free.

Three gated runs in dependency order:

| Stage | Surface | Wall time | Cost | Gate |
|---|---|---|---|---|
| **A** Smoke | 50 stratified rows | ~1 min | $0 (Groq free tier) | always on |
| **B** Golden | 234 accepted golden rows | ~5 min | $0 | `RUN_GOLDEN = True` |
| **C** Full | 12,723 medqa_4opt rows | ~5 h | $0 | `RUN_FULL = False` (flip to True when A and B look right) |

Each stage writes `results/exp_01_base_llm__<surface>/` with `predictions.jsonl`, `retrieval.jsonl` (empty for No-RAG), and `summary.json`. The runner is **resumable** — re-running any stage skips already-completed `question_id`s.

Per [`docs/results_schema.md`](../docs/results_schema.md), the four `RAGAS_*` columns and `Recall@K` ship as `null` here; they get filled by `src/eval/ragas_eval.py` (next session) for golden-234 only.

---

Tables this notebook fills: **Table 1** (row 1 — `Acuuracy`, `Exact Match`, `Generator_Model`, `mean_latency_s`) · **Table 9** (col 1 — No-RAG baseline). RAGAS rows of those tables come later.

## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

# Make `src` importable from a notebook (repo root one level up).
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

from src.data.loaders import load_golden, load_medqa_4opt
from src.eval.runner import run_experiment
from src.retrieval.none import NoRetrieval

import os

print("GROQ_API_KEY present:", "\u2713" if os.environ.get("GROQ_API_KEY") else "\u2717")
print("Repo root:", REPO_ROOT)

GROQ_API_KEY present: ✓
Repo root: /Users/rajak/Workstation/Projects/myGitHub/thesis-project


## 2. Configuration

`SMOKE_N` is small enough to flush bugs cheaply; `RUN_GOLDEN` and `RUN_FULL` are flags so you can step through stages one at a time.

In [6]:
EXPERIMENT_ID = "EXP_01_BASE_LLM"
RESULTS_DIR = REPO_ROOT / "results"

SMOKE_N = 50
SMOKE_SEED = 42

# Stage flags — flip these on once the previous stage looks right.
RUN_SMOKE = True       # Stage A · ~1 min
RUN_GOLDEN = True      # Stage B · ~5 min
RUN_FULL = True       # Stage C · ~5 h Groq — DON'T flip until A and B are clean

RETRIEVER = NoRetrieval()  # No-RAG by definition for EXP_01

## 3. Stage A — Smoke (50 stratified rows)

Stratifies by `meta_info` (step1 / step2&3) so the smoke surface mirrors the full corpus's 55/45 split. Seeded → reproducible.

In [3]:
if RUN_SMOKE:
    medqa = load_medqa_4opt()

    # Stratified sample by meta_info, weighted to match the corpus 55/45 split.
    smoke = (
        medqa.groupby("meta_info", group_keys=False)
        .apply(lambda g: g.sample(n=max(1, round(SMOKE_N * len(g) / len(medqa))), random_state=SMOKE_SEED))
        .reset_index(drop=True)
    )
    # Trim or pad to exactly SMOKE_N
    smoke = smoke.head(SMOKE_N) if len(smoke) >= SMOKE_N else medqa.sample(n=SMOKE_N, random_state=SMOKE_SEED).reset_index(drop=True)

    print(f"Smoke surface: {len(smoke)} rows | meta_info mix: {dict(smoke['meta_info'].value_counts())}")

    smoke_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=smoke,
        output_dir=RESULTS_DIR / "exp_01_base_llm__smoke_50",
        experiment_id=EXPERIMENT_ID,
        dataset_label="smoke_50",
    )
    print(json.dumps(smoke_summary, indent=2))
else:
    print("RUN_SMOKE = False — skipping Stage A")

Smoke surface: 50 rows | meta_info mix: {'step1': np.int64(28), 'step2&3': np.int64(22)}


/var/folders/q2/0276zgxs0m7bj83vqzpkxlqh0000gn/T/ipykernel_42260/2406690293.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=max(1, round(SMOKE_N * len(g) / len(medqa))), random_state=SMOKE_SEED))


EXP_01_BASE_LLM:   0%|          | 0/50 [00:00<?, ?it/s]

{
  "experiment_id": "EXP_01_BASE_LLM",
  "dataset": "smoke_50",
  "n_questions": 50,
  "Generator_Model": "llama-3.3-70b-versatile",
  "temperature": 0.0,
  "max_tokens": 700,
  "Acuuracy": 0.94,
  "Exact_Match": 0.94,
  "n_correct": 47,
  "Recall@3": null,
  "Recall@5": null,
  "Recall@10": null,
  "MRR": null,
  "RAGAS_Faithfulness": null,
  "RAGAS_Hallucination_Rate": null,
  "RAGAS_Answer_Relevance": null,
  "RAGAS_Context_Precision": null,
  "RAGAS_Context_Recall": null,
  "Answer_Correctness": null,
  "mean_latency_s": 0.2717434787750244,
  "wall_time_s_this_run": 13.70678186416626,
  "n_calls_this_run": 50,
  "cache_hits_this_run": 0,
  "cache_hit_rate_this_run": 0.0,
  "timestamp_utc": "2026-05-05T13:56:51Z"
}


**Acceptance gate (Stage A):**
- `n_questions` == 50
- `Acuuracy` ≥ 0.50 (LLaMA 3.3 70B's known No-RAG ceiling on MedQA is ~0.75; 0.50 floor catches gross prompt regressions)
- `mean_latency_s` < 5.0 (Notebook 03 measured 1.38 s)
- `cache_hit_rate_this_run` is 0.0 on the first run, ~1.0 on a re-run

If anything looks wrong, fix and re-run — the cache means iterations are free.

## 4. Stage B — Golden 234

Same retriever, different surface. The 234 accepted rows from Phase 3. Cache will hit on any rows that overlap with Stage A (rare unless the same `question` text appears in both).

In [4]:
if RUN_GOLDEN:
    golden = load_golden()
    print(f"Golden surface: {len(golden)} accepted rows")

    golden_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=golden,
        output_dir=RESULTS_DIR / "exp_01_base_llm__golden_234",
        experiment_id=EXPERIMENT_ID,
        dataset_label="golden_234",
    )
    print(json.dumps(golden_summary, indent=2))
else:
    print("RUN_GOLDEN = False — skipping Stage B")

Golden surface: 234 accepted rows


EXP_01_BASE_LLM:   0%|          | 0/234 [00:00<?, ?it/s]

{
  "experiment_id": "EXP_01_BASE_LLM",
  "dataset": "golden_234",
  "n_questions": 234,
  "Generator_Model": "llama-3.3-70b-versatile",
  "temperature": 0.0,
  "max_tokens": 700,
  "Acuuracy": 0.9017094017094017,
  "Exact_Match": 0.9017094017094017,
  "n_correct": 211,
  "Recall@3": null,
  "Recall@5": null,
  "Recall@10": null,
  "MRR": null,
  "RAGAS_Faithfulness": null,
  "RAGAS_Hallucination_Rate": null,
  "RAGAS_Answer_Relevance": null,
  "RAGAS_Context_Precision": null,
  "RAGAS_Context_Recall": null,
  "Answer_Correctness": null,
  "mean_latency_s": 0.24914994606604943,
  "wall_time_s_this_run": 57.74756908416748,
  "n_calls_this_run": 234,
  "cache_hits_this_run": 3,
  "cache_hit_rate_this_run": 0.01282051282051282,
  "timestamp_utc": "2026-05-05T13:58:44Z"
}


**Acceptance gate (Stage B):** `n_questions` == 234 · `Acuuracy` in roughly the same range as smoke (the golden subset is stratified, so accuracy should be close to the full-surface number ± a few points).

## 5. Stage C — Full 12,723 (gated · ~5 h)

**Set `RUN_FULL = True` in the config cell only after Stages A and B look right.** This is the long run that actually fills Table 1.

**Before flipping:** `caffeinate -dimsu &` to keep the Mac awake, plug in, stable Wi-Fi. The runner is resumable, so a mid-run failure (rate limit, network hiccup) just means re-running — already-done questions skip via `predictions.jsonl`.

In [7]:
if RUN_FULL:
    medqa = load_medqa_4opt()
    print(f"Full surface: {len(medqa)} rows")

    full_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=medqa,
        output_dir=RESULTS_DIR / "exp_01_base_llm__full_12723",
        experiment_id=EXPERIMENT_ID,
        dataset_label="full_12723",
    )
    print(json.dumps(full_summary, indent=2))
else:
    print("RUN_FULL = False — skipping Stage C (set RUN_FULL = True in the config cell when ready)")

Full surface: 12723 rows


EXP_01_BASE_LLM:   0%|          | 0/12723 [00:00<?, ?it/s]

{
  "experiment_id": "EXP_01_BASE_LLM",
  "dataset": "full_12723",
  "n_questions": 12723,
  "Generator_Model": "llama-3.3-70b-versatile",
  "temperature": 0.0,
  "max_tokens": 700,
  "Acuuracy": 0.8692918336870235,
  "Exact_Match": 0.8692918336870235,
  "n_correct": 11060,
  "Recall@3": null,
  "Recall@5": null,
  "Recall@10": null,
  "MRR": null,
  "RAGAS_Faithfulness": null,
  "RAGAS_Hallucination_Rate": null,
  "RAGAS_Answer_Relevance": null,
  "RAGAS_Context_Precision": null,
  "RAGAS_Context_Recall": null,
  "Answer_Correctness": null,
  "mean_latency_s": 0.27909377437712385,
  "wall_time_s_this_run": 3502.2406690120697,
  "n_calls_this_run": 12723,
  "cache_hits_this_run": 286,
  "cache_hit_rate_this_run": 0.02247897508449265,
  "timestamp_utc": "2026-05-05T15:00:47Z"
}


## 6. Inspect predictions

Quick eyeball on the smoke run — first few wrong answers tell you whether the prompt path is sane (e.g. parsing letters correctly, not getting cut off by `max_tokens`).

In [8]:
import pandas as pd

smoke_pred_path = RESULTS_DIR / "exp_01_base_llm__smoke_50" / "predictions.jsonl"
if smoke_pred_path.exists():
    preds = pd.DataFrame(json.loads(line) for line in smoke_pred_path.read_text().splitlines())
    print(f"Smoke: {preds['is_correct'].sum()} / {len(preds)} correct = {preds['is_correct'].mean():.1%}")
    print("Predicted-letter distribution:", dict(preds["pred_letter"].value_counts(dropna=False)))
    print("\nFirst 5 wrong answers (informative for prompt debugging):")
    wrong = preds[~preds["is_correct"]].head(5)
    for _, r in wrong.iterrows():
        raw = (r["raw_response"] or "").replace("\n", " ")[:80]
        print(f"  {r['question_id']}: gold={r['gold_letter']} pred={r['pred_letter']} | raw[:80]={raw!r}")
else:
    print("No smoke predictions.jsonl yet — run Stage A first.")

Smoke: 47 / 50 correct = 94.0%
Predicted-letter distribution: {'B': np.int64(17), 'A': np.int64(14), 'D': np.int64(10), 'C': np.int64(9)}

First 5 wrong answers (informative for prompt debugging):
  medqa_03822: gold=C pred=B | raw[:80]='B'
  medqa_06968: gold=A pred=B | raw[:80]='B'
  medqa_12358: gold=C pred=B | raw[:80]='B'


## 7. Excel-paste row

When all three stages have run, the `summary.json` files are the paste-into-workbook rows. EXP_01 contributes one row to **Table 1** (full surface) and one set of RAGAS values to **Table 1** golden columns (filled later by the RAGAS judge).

In [9]:
for label in ("smoke_50", "golden_234", "full_12723"):
    p = RESULTS_DIR / f"exp_01_base_llm__{label}" / "summary.json"
    if p.exists():
        s = json.loads(p.read_text())
        print(f"\n=== {label} ===")
        print(f"  n_questions      : {s['n_questions']}")
        print(f"  Acuuracy         : {s['Acuuracy']:.4f}")
        print(f"  Exact_Match      : {s['Exact_Match']:.4f}")
        print(f"  mean_latency_s   : {s['mean_latency_s']:.2f}")
        print(f"  Generator_Model  : {s['Generator_Model']}")
        print(f"  timestamp_utc    : {s['timestamp_utc']}")
    else:
        print(f"\n=== {label} === (no summary.json yet)")


=== smoke_50 ===
  n_questions      : 50
  Acuuracy         : 0.9400
  Exact_Match      : 0.9400
  mean_latency_s   : 0.27
  Generator_Model  : llama-3.3-70b-versatile
  timestamp_utc    : 2026-05-05T13:56:51Z

=== golden_234 ===
  n_questions      : 234
  Acuuracy         : 0.9017
  Exact_Match      : 0.9017
  mean_latency_s   : 0.25
  Generator_Model  : llama-3.3-70b-versatile
  timestamp_utc    : 2026-05-05T13:58:44Z

=== full_12723 ===
  n_questions      : 12723
  Acuuracy         : 0.8693
  Exact_Match      : 0.8693
  mean_latency_s   : 0.28
  Generator_Model  : llama-3.3-70b-versatile
  timestamp_utc    : 2026-05-05T15:00:47Z


---

**Next session.** Build `src/eval/ragas_eval.py` (Claude 3.5 Sonnet judge) and run it against `results/exp_01_base_llm__golden_234/predictions.jsonl` to fill the four `RAGAS_*` columns + `Answer_Correctness` for EXP_01. Then EXP_02 (Naive RAG) reuses this same notebook structure with `NaiveRetriever` instead of `NoRetrieval`.